# ARC2 inference-only hybrid candidate

Public Apache-2.0 Sorokin grid model. Demonstration-consistent symbolic candidates retain first-attempt priority. Identity and transpose decoding, rectangular grammar, coverage-first scheduling and chunked KV-cached prefill with a baseline parity gate. No adaptation or answer-file access. Bounded visible-input preview; full-budget official rerun. Training-development hybrid: 281/1076, potentially contaminated by checkpoint training; no held-out improvement claimed.

In [ ]:
from pathlib import Path
import json,time,sys,os
START=time.monotonic()
sources={'arc2/__init__.py': '"""Separate ARC-AGI-2 experiments; no dependency on the ARC-AGI-3 runtime."""\n', 'arc2/symbolic.py': '"""Bounded task-local program search; uses only the task\'s demonstration pairs."""\nfrom collections import Counter,deque\nimport time\n\n\ndef grid(value):\n    if not isinstance(value,(list,tuple)) or not 1<=len(value)<=30:raise ValueError(\'Invalid grid height\')\n    rows=tuple(tuple(row) for row in value)\n    if not rows[0] or len(rows[0])>30 or any(len(row)!=len(rows[0]) for row in rows):raise ValueError(\'Invalid grid width\')\n    if any(type(c) is not int or not 0<=c<=9 for row in rows for c in row):raise ValueError(\'Invalid grid colors\')\n    return rows\n\n\ndef modal(g):return Counter(c for row in g for c in row).most_common(1)[0][0]\n\n\ndef rotate(g):return tuple(tuple(row) for row in zip(*g[::-1]))\n\n\ndef orient(g,index):\n    if index>=4:g=tuple(tuple(row) for row in zip(*g));index-=4\n    for _ in range(index):g=rotate(g)\n    return g\n\n\ndef crop(g,color=None,background=None):\n    bg=modal(g) if background is None else background\n    points=[(r,c) for r,row in enumerate(g) for c,x in enumerate(row) if (x==color if color is not None else x!=bg)]\n    if not points:return None\n    r0=min(p[0] for p in points);r1=max(p[0] for p in points)+1\n    c0=min(p[1] for p in points);c1=max(p[1] for p in points)+1\n    return tuple(row[c0:c1] for row in g[r0:r1])\n\n\ndef object_crop(g,selection,monochrome,background):\n    bg=modal(g) if background is None else background\n    seen=set();objects=[];h=len(g);w=len(g[0])\n    for r in range(h):\n        for c in range(w):\n            if (r,c) in seen or g[r][c]==bg:continue\n            pending=deque([(r,c)]);seen.add((r,c));points=[]\n            while pending:\n                y,x=pending.popleft();points.append((y,x))\n                for yy,xx in ((y-1,x),(y+1,x),(y,x-1),(y,x+1)):\n                    if 0<=yy<h and 0<=xx<w and (yy,xx) not in seen and g[yy][xx]!=bg and (not monochrome or g[yy][xx]==g[r][c]):\n                        seen.add((yy,xx));pending.append((yy,xx))\n            objects.append(points)\n    if not objects:return None\n    objects.sort(key=lambda p:(len(p),min(p)))\n    points=objects[-1] if selection==\'largest\' else objects[0]\n    r0=min(p[0] for p in points);r1=max(p[0] for p in points)+1\n    c0=min(p[1] for p in points);c1=max(p[1] for p in points)+1\n    return tuple(row[c0:c1] for row in g[r0:r1])\n\n\ndef resize(g,scale,repeat=False):\n    if len(g)*scale>30 or len(g[0])*scale>30:return None\n    if repeat:return tuple(row*scale for _ in range(scale) for row in g)\n    return tuple(tuple(c for x in row for c in [x]*scale) for row in g for _ in range(scale))\n\n\ndef reduce_grid(g,scale):\n    if len(g)%scale or len(g[0])%scale:return None\n    result=[]\n    for r in range(0,len(g),scale):\n        row=[]\n        for c in range(0,len(g[0]),scale):\n            values={g[y][x] for y in range(r,r+scale) for x in range(c,c+scale)}\n            if len(values)!=1:return None\n            row.append(next(iter(values)))\n        result.append(tuple(row))\n    return tuple(result)\n\n\ndef compress(g):\n    rows=[]\n    for row in g:\n        if not rows or row!=rows[-1]:rows.append(row)\n    columns=[]\n    for col in zip(*rows):\n        if not columns or col!=columns[-1]:columns.append(col)\n    return tuple(tuple(row) for row in zip(*columns))\n\n\ndef halves(g,axis,mode,background):\n    if axis:g=orient(g,4)\n    n=len(g);middle=n//2\n    if n<2:return None\n    first=g[:middle];second=g[middle+n%2:]\n    bg=modal(g) if background is None else background\n    if mode==\'first\':out=first\n    elif mode==\'second\':out=second\n    else:\n        operations={\'or\':lambda a,b:a or b,\'and\':lambda a,b:a and b,\'xor\':lambda a,b:a!=b}\n        op=operations[mode]\n        out=tuple(tuple(int(op(a!=bg,b!=bg)) for a,b in zip(ra,rb)) for ra,rb in zip(first,second))\n    return orient(out,4) if axis else out\n\n\ndef operations(colors):\n    yield \'identity\',0,lambda g:g\n    yield \'compress_repeated_rows_columns\',1,compress\n    for bg in (None,0):\n        yield f\'foreground_bbox_{bg}\',1,lambda g,bg=bg:crop(g,background=bg)\n        for selection in (\'largest\',\'smallest\'):\n            for mono in (False,True):\n                yield f\'{selection}_object_{mono}_{bg}\',2,lambda g,s=selection,m=mono,b=bg:object_crop(g,s,m,b)\n        for axis in (0,1):\n            for mode in (\'first\',\'second\',\'or\',\'and\',\'xor\'):\n                yield f\'halves_{axis}_{mode}_{bg}\',2,lambda g,a=axis,m=mode,b=bg:halves(g,a,m,b)\n    for color in colors:yield f\'color_bbox_{color}\',2,lambda g,c=color:crop(g,color=c)\n    for scale in (2,3,4):\n        yield f\'upscale_{scale}\',1,lambda g,s=scale:resize(g,s)\n        yield f\'tile_{scale}\',2,lambda g,s=scale:resize(g,s,True)\n        yield f\'uniform_downscale_{scale}\',1,lambda g,s=scale:reduce_grid(g,s)\n\n\ndef fit_mapping(inputs,outputs):\n    mapping={}\n    for before,after in zip(inputs,outputs):\n        if before is None or len(before)!=len(after) or len(before[0])!=len(after[0]):return None\n        for ra,rb in zip(before,after):\n            for a,b in zip(ra,rb):\n                if a in mapping and mapping[a]!=b:return None\n                mapping[a]=b\n    return mapping\n\n\ndef solve(task,seconds=1.0):\n    started=time.monotonic();examples=task[\'train\'];tests=[grid(t[\'input\']) for t in task[\'test\']]\n    if not examples:return [dict(attempt_1=[list(row) for row in g],attempt_2=[list(row) for row in g]) for g in tests],dict(programs=0,reason=\'no demonstrations\')\n    inputs=[grid(t[\'input\']) for t in examples];outputs=[grid(t[\'output\']) for t in examples]\n    colors=sorted({c for g in inputs for row in g for c in row})\n    mappings={};prediction_sets=set();candidates=[];tried=0;timed_out=False\n    for name,cost,operation in operations(colors):\n        if time.monotonic()-started>=seconds:timed_out=True;break\n        transformed=[operation(g) for g in inputs]\n        if any(g is None for g in transformed):continue\n        for orientation in range(8):\n            views=tuple(orient(g,orientation) for g in transformed)\n            tried+=1\n            if views not in mappings:mappings[views]=fit_mapping(views,outputs)\n            mapping=mappings[views]\n            if mapping is None:continue\n            predictions=[]\n            for g in tests:\n                result=operation(g)\n                if result is None:break\n                result=orient(result,orientation)\n                result=tuple(tuple(mapping.get(c,c) for c in row) for row in result)\n                predictions.append(result)\n            if len(predictions)!=len(tests):continue\n            signature=tuple(predictions)\n            if signature in prediction_sets:continue\n            prediction_sets.add(signature)\n            recolors=sum(a!=b for a,b in mapping.items())\n            score=cost+0.1*(orientation!=0)+0.05*recolors\n            candidates.append((score,name,orientation,tuple(predictions)))\n    candidates.sort(key=lambda c:(c[0],c[1],c[2]))\n    attempts=[]\n    for i,g in enumerate(tests):\n        unique=[]\n        for candidate in candidates:\n            prediction=candidate[3][i]\n            if prediction not in unique:unique.append(prediction)\n            if len(unique)==2:break\n        if not unique:unique=[g]\n        if len(unique)==1:unique.append(g if g!=unique[0] else tuple(tuple(0 for _ in row) for row in g))\n        attempts.append(dict(attempt_1=[list(row) for row in unique[0]],attempt_2=[list(row) for row in unique[1]]))\n    return attempts,dict(programs=tried,consistent_programs=len(candidates),timed_out=timed_out,\n        elapsed_seconds=time.monotonic()-started,selected=[dict(name=c[1],orientation=c[2],complexity=c[0]) for c in candidates[:2]])\n', 'arc2/evaluation.py': '"""Official two-attempt exact-match metric, averaged over test outputs."""\nfrom .symbolic import grid\n\n\ndef validate_submission(challenges,submission):\n    if set(challenges)!=set(submission):raise ValueError(\'Submission IDs must exactly match challenge IDs\')\n    for key,task in challenges.items():\n        guesses=submission[key]\n        if not isinstance(guesses,list) or len(guesses)!=len(task[\'test\']):raise ValueError(\'Test-output count mismatch: \'+key)\n        for attempt in guesses:\n            if not isinstance(attempt,dict) or set(attempt)!=set((\'attempt_1\',\'attempt_2\')):\n                raise ValueError(\'Exactly two named attempts required\')\n            grid(attempt[\'attempt_1\']);grid(attempt[\'attempt_2\'])\n\n\ndef score_submission(challenges,solutions,submission):\n    validate_submission(challenges,submission)\n    if set(challenges)!=set(solutions):raise ValueError(\'Solution IDs differ\')\n    results=[]\n    for key,task in challenges.items():\n        if len(solutions[key])!=len(task[\'test\']):raise ValueError(\'Solution-output count mismatch\')\n        for index,target in enumerate(solutions[key]):\n            target=grid(target);attempt=submission[key][index]\n            correct=any(grid(attempt[name])==target for name in (\'attempt_1\',\'attempt_2\'))\n            results.append(dict(task_id=key,test_index=index,correct=correct))\n    return dict(correct_outputs=sum(r[\'correct\'] for r in results),test_outputs=len(results),\n        score=100*sum(r[\'correct\'] for r in results)/len(results) if results else 0,per_output=results)\n', 'arc2/neural.py': '"""Inference-only adapter for the public NVARC-derived 4B grid checkpoint.\n\nUses the public checkpoint\'s digit/chat serialization. This does not reproduce\nNVARC\'s LoRA test-time training, DFS decoding or augmentation rescoring.\n"""\nimport time\n\nfrom .symbolic import grid,orient\n\n\ndef grid_text(value):return \'\\n\'.join(\'\'.join(map(str,row)) for row in grid(value))\n\n\ndef prompt(task,index,orientation=0,skip=0):\n    parts=[]\n    for pair in task[\'train\'][skip:]:\n        parts.extend((\'<|im_start|>user\\n\',grid_text(orient(grid(pair[\'input\']),orientation)),\n            \'<|im_end|><|im_start|>assistant\\n\',grid_text(orient(grid(pair[\'output\']),orientation)),\'<|im_end|>\'))\n    parts.extend((\'<|im_start|>user\\n\',grid_text(orient(grid(task[\'test\'][index][\'input\']),orientation)),\n        \'<|im_end|><|im_start|>assistant\\n\'))\n    return \'\'.join(parts)\n\n\ndef parse_output(text):\n    text=text.split(\'<|im_end|>\',1)[0].strip(\'\\n\')\n    if not text or any(c not in \'0123456789\\n\' for c in text):raise ValueError(\'Expected digit grid only\')\n    return grid([[int(c) for c in row] for row in text.split(\'\\n\')])\n\n\ndef allowed_tokens(suffix,digits,newline,eos):\n    if eos in suffix:return [eos]\n    rows=[];current=[]\n    for token in suffix:\n        if token==newline:rows.append(current);current=[]\n        elif token in digits:current.append(token)\n        else:raise ValueError(\'Invalid generated grid token\')\n    if not rows:\n        result=list(digits) if len(current)<30 else []\n        if current:result.extend((newline,eos))\n        return result\n    width=len(rows[0])\n    if not 1<=width<=30 or any(len(row)!=width for row in rows):raise ValueError(\'Nonrectangular generated prefix\')\n    if len(current)<width:return list(digits)\n    if len(current)>width:raise ValueError(\'Generated row too wide\')\n    return [eos]+([newline] if len(rows)<29 else [])\n\n\nclass GridModel:\n    def __init__(self,path):\n        import torch\n        from transformers import AutoModelForCausalLM,AutoTokenizer\n        if not torch.cuda.is_available():raise RuntimeError(\'Actual CUDA GPU required before loading neural model\')\n        self.torch=torch\n        self.tokenizer=AutoTokenizer.from_pretrained(str(path),local_files_only=True,trust_remote_code=False)\n        self.model=AutoModelForCausalLM.from_pretrained(str(path),local_files_only=True,trust_remote_code=False,\n            use_safetensors=True,torch_dtype=torch.float16,device_map={\'\':0},low_cpu_mem_usage=True,\n            attn_implementation=\'sdpa\')\n        self.model.eval()\n        self.digits=[self.one_token(str(c)) for c in range(10)]\n        self.newline=self.one_token(\'\\n\');self.eos=self.tokenizer.eos_token_id\n        if self.eos is None:raise ValueError(\'Tokenizer has no EOS\')\n        ids=self.digits+[self.newline,self.eos]\n        if len(set(ids))!=12 or max(ids)>=self.model.config.vocab_size:\n            raise ValueError(\'Checkpoint grid-token vocabulary mismatch\')\n\n    def one_token(self,text):\n        ids=self.tokenizer.encode(text,add_special_tokens=False)\n        if len(ids)!=1:raise ValueError(\'Checkpoint requires atomic grid tokens\')\n        return ids[0]\n\n    def generate(self,task,index,orientation=0,deadline=None):\n        if orientation not in (0,4):raise ValueError(\'Only identity/transpose inference supported\')\n        if deadline is not None and time.monotonic()>=deadline-5:return None,dict(reason=\'budget\')\n        started=time.monotonic();skip=0\n        while True:\n            ids=self.tokenizer.encode(prompt(task,index,orientation,skip),add_special_tokens=False)\n            if len(ids)+960<=8192:break\n            if skip>=len(task[\'train\']):return None,dict(reason=\'context budget\')\n            skip+=1\n        if max(ids)>=self.model.config.vocab_size:raise ValueError(\'Prompt token exceeds checkpoint vocabulary\')\n        tokens=self.torch.tensor([ids],device=self.model.device,dtype=self.torch.long)\n        options=dict(max_new_tokens=960,do_sample=False,use_cache=True,eos_token_id=self.eos,pad_token_id=self.eos,\n            prefix_allowed_tokens_fn=lambda batch,seq:allowed_tokens(seq[len(ids):].tolist(),self.digits,self.newline,self.eos))\n        if deadline is not None:options[\'max_time\']=max(1.0,deadline-time.monotonic())\n        try:\n            with self.torch.inference_mode():output=self.model.generate(input_ids=tokens,**options)\n        except self.torch.OutOfMemoryError:\n            self.torch.cuda.empty_cache()\n            return None,dict(reason=\'GPU memory exhausted\',prompt_tokens=len(ids),orientation=orientation)\n        suffix=output[0,len(ids):].tolist()\n        info=dict(prompt_tokens=len(ids),generated_tokens=len(suffix),orientation=orientation,\n            demonstrations_omitted=skip,elapsed_seconds=time.monotonic()-started,terminated_with_eos=bool(suffix and suffix[-1]==self.eos))\n        if not info[\'terminated_with_eos\']:return None,dict(**info,reason=\'unfinished output\')\n        try:result=parse_output(self.tokenizer.decode(suffix,skip_special_tokens=False))\n        except ValueError:return None,dict(**info,reason=\'invalid grid\')\n        if orientation==4:result=orient(result,4)\n        return [list(row) for row in result],info\n', 'arc2/bounded_inference.py': '"""Chunked prefill and coverage-first inference, separate from frozen baselines."""\nimport time\n\nfrom .neural import allowed_tokens, parse_output, prompt\nfrom .symbolic import grid, orient, solve\nfrom .evaluation import validate_submission\n\n\ndef select_hybrid(floor, consistent, candidates):\n    """Retain the measured baseline selection rule and distinct second guess."""\n    choices = []\n    for candidate in candidates:\n        if candidate not in choices:\n            choices.append(candidate)\n    if not choices:\n        choices = [floor[\'attempt_1\']]\n    if len(choices) == 1:\n        choices.append(floor[\'attempt_2\'])\n    first = floor[\'attempt_1\'] if consistent else choices[0]\n    second = next((c for c in choices+[floor[\'attempt_2\']] if c != first), first)\n    return dict(attempt_1=first, attempt_2=second)\n\n\ndef decode_tokens(model, ids, deadline, chunk_size=256):\n    """Causal KV-cached greedy decoding; never allocate a full prompt attention map."""\n    torch = model.torch\n    cache = None\n    suffix = []\n    with torch.inference_mode():\n        for start in range(0, len(ids), chunk_size):\n            if time.monotonic() >= deadline:\n                return suffix, \'prefill deadline\'\n            tokens = torch.tensor([ids[start:start+chunk_size]], device=model.model.device, dtype=torch.long)\n            output = model.model(input_ids=tokens, past_key_values=cache, use_cache=True)\n            cache = output.past_key_values\n            logits = output.logits[0, -1].clone()\n            del output\n        for _ in range(960):\n            if time.monotonic() >= deadline:\n                return suffix, \'decode deadline\'\n            allowed = allowed_tokens(suffix, model.digits, model.newline, model.eos)\n            # Preserve the raw model ranking among grammar-allowed tokens.\n            token = allowed[int(logits[allowed].argmax().item())]\n            suffix.append(token)\n            if token == model.eos:\n                return suffix, \'complete\'\n            output = model.model(input_ids=torch.tensor([[token]], device=model.model.device, dtype=torch.long),\n                                 past_key_values=cache, use_cache=True)\n            cache = output.past_key_values\n            logits = output.logits[0, -1].clone()\n            del output\n    return suffix, \'token limit\'\n\n\ndef generate_chunked(model, task, index, orientation=0, deadline=None, chunk_size=256):\n    if orientation not in (0, 4):\n        raise ValueError(\'Only measured identity/transpose views supported\')\n    started = time.monotonic()\n    deadline = deadline if deadline is not None else started+120\n    if started >= deadline-1:\n        return None, dict(reason=\'budget\', orientation=orientation)\n    skip = 0\n    while True:\n        ids = model.tokenizer.encode(prompt(task, index, orientation, skip), add_special_tokens=False)\n        if len(ids)+960 <= 8192:\n            break\n        if skip >= len(task[\'train\']):\n            return None, dict(reason=\'context budget\', orientation=orientation)\n        skip += 1\n    if not ids or max(ids) >= model.model.config.vocab_size:\n        raise ValueError(\'Prompt vocabulary mismatch\')\n    info = dict(prompt_tokens=len(ids), orientation=orientation, demonstrations_omitted=skip,\n                chunk_size=chunk_size)\n    try:\n        suffix, stop = decode_tokens(model, ids, deadline, chunk_size)\n    except model.torch.OutOfMemoryError:\n        # decode_tokens has unwound, releasing its live tensors and cache first.\n        model.torch.cuda.empty_cache()\n        return None, dict(**info, reason=\'GPU memory exhausted\', elapsed_seconds=time.monotonic()-started)\n    info.update(generated_tokens=len(suffix), stop_reason=stop, elapsed_seconds=time.monotonic()-started,\n                terminated_with_eos=bool(suffix and suffix[-1] == model.eos))\n    if not info[\'terminated_with_eos\']:\n        return None, dict(**info, reason=\'unfinished output\')\n    try:\n        result = parse_output(model.tokenizer.decode(suffix, skip_special_tokens=False))\n    except ValueError:\n        return None, dict(**info, reason=\'invalid grid\')\n    if orientation == 4:\n        result = orient(result, 4)\n    return [list(row) for row in result], info\n\n\ndef run(challenges, model, deadline, save, generator=generate_chunked, clock=time.monotonic):\n    """Complete fallback first, then one view across all outputs before the next."""\n    floors = {}\n    consistent = {}\n    candidates = {}\n    submission = {}\n    events = []\n    for key, task in sorted(challenges.items()):\n        floor, info = solve(task, seconds=0.25 if clock() < deadline else 0)\n        floors[key] = floor\n        consistent[key] = info.get(\'consistent_programs\', 0) > 0\n        candidates[key] = [[] for _ in task[\'test\']]\n        submission[key] = [dict(attempt) for attempt in floor]\n    validate_submission(challenges, submission)\n    save(submission, events)\n    work = [(key, i) for key in sorted(challenges) for i in range(len(challenges[key][\'test\']))]\n    for view in (0, 4):\n        for position, (key, index) in enumerate(work):\n            remaining = deadline-clock()\n            if remaining <= 2:\n                break\n            # Equal opportunity for every remaining output, capped at 120s/view.\n            share = min(120, max(2, remaining/(len(work)-position)))\n            prediction, info = generator(model, challenges[key], index, orientation=view,\n                                         deadline=min(deadline, clock()+share))\n            events.append(dict(task_id=key, test_index=index, **info))\n            if prediction is not None:\n                candidates[key][index].append(prediction)\n            submission[key][index] = select_hybrid(floors[key][index], consistent[key], candidates[key][index])\n            validate_submission(challenges, submission)\n            save(submission, events)\n    return submission, events\n'}

RERUN=os.environ.get('KAGGLE_IS_COMPETITION_RERUN','').strip().lower() in ('1','true')
END=START+(43200-600 if RERUN else 560)
for name,source in sources.items():
    p=Path('/kaggle/working')/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_text(source)
sys.path.insert(0,'/kaggle/working')
from arc2.symbolic import solve
from arc2.evaluation import validate_submission
from arc2.neural import GridModel
from arc2.bounded_inference import generate_chunked,run
import torch,transformers
roots=[Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2'),Path('/kaggle/input/arc-prize-2026-arc-agi-2')]
root=next((p for p in roots if (p/'arc-agi_test_challenges.json').is_file()),None)
if root is None:raise FileNotFoundError('Actual competition test inputs not mounted')
challenges=json.loads((root/'arc-agi_test_challenges.json').read_text())
submission={key:solve(task,seconds=0.25)[0] for key,task in sorted(challenges.items())}
events=[]
def save(predictions,records):
    submission.update(predictions)
    validate_submission(challenges,submission)
    p=Path('/kaggle/working/submission.json.tmp');p.write_text(json.dumps(submission));p.replace(p.with_suffix(''))
    Path('/kaggle/working/inference-events.json').write_text(json.dumps(records))
save({},events)
models=[Path('/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1'),Path('/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1')]
model_path=next((p for p in models if (p/'config.json').is_file()),None)
if model_path is None:raise FileNotFoundError('Pinned public checkpoint missing')
model=GridModel(model_path)
print('ARC2_MODEL_READY',dict(gpu=torch.cuda.get_device_name(0),torch=torch.__version__,transformers=transformers.__version__,rerun=RERUN),flush=True)
# Fixed public TRAINING inputs chosen by event failure/length, never by labels.
# Small-prompt numerical parity must hold before using the changed decoder.
training=json.loads((root/'arc-agi_training_challenges.json').read_text())
checks=[]
for key,view in [('00576224',0),('007bbfb7',4)]:
    task=training[key]
    before,binfo=model.generate(task,0,view,deadline=min(END,time.monotonic()+25))
    torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    after,ainfo=generate_chunked(model,task,0,view,deadline=min(END,time.monotonic()+25))
    checks.append(dict(task_id=key,orientation=view,equal=before is not None and before==after,
                       baseline=binfo,chunked=ainfo,peak_bytes=torch.cuda.max_memory_allocated()))
    print('PARITY',json.dumps(checks[-1]),flush=True)
use_chunked=all(c['equal'] for c in checks)
memory_check=None
if not RERUN and use_chunked and time.monotonic()<END-150:
    torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats()
    prediction,info=generate_chunked(model,training['05a7bcf2'],0,deadline=min(END-80,time.monotonic()+150))
    memory_check=dict(task_id='05a7bcf2',valid=prediction is not None,info=info,
                      peak_bytes=torch.cuda.max_memory_allocated())
    print('LONG_PROMPT_CHECK',json.dumps(memory_check),flush=True)
def baseline_generator(m,task,index,orientation,deadline):
    return m.generate(task,index,orientation,deadline)
# Visible preview uses a small actual-test subset to fit the remaining free quota.
# Official rerun processes ALL supplied test inputs; fallback was written first.
active=challenges if RERUN else {k:challenges[k] for k in sorted(challenges)[:4]}
_,events=run(active,model,END,save,generator=generate_chunked if use_chunked else baseline_generator)
save({},events)
report=dict(rerun=RERUN,elapsed_seconds=time.monotonic()-START,tasks=len(challenges),
    test_outputs=sum(len(t['test']) for t in challenges.values()),active_neural_tasks=len(active),
    valid_generations=sum(not e.get('reason') for e in events),events=len(events),
    decoder='chunked' if use_chunked else 'baseline',parity_checks=checks,memory_check=memory_check,
    boundary='Actual competition-input inference. Visible preview capped below ten minutes; official rerun capped below twelve hours. No solution files read. Functional and memory diagnostics do not establish accuracy or an official score.')
Path('/kaggle/working/competition-run.json').write_text(json.dumps(report,indent=2))
print('ARC2_COMPETITION_COMPLETE',json.dumps(report),flush=True)
